## Step 5: SVM Classification
We train a Support Vector Machine (SVM) to classify:
- Left fist imagination
- Right fist imagination
- Rest  imagination

SVM finds the best boundary line between two classes.
We only use left and right fist — not rest — for now.

In [47]:
import mne
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

In [48]:
epochs = mne.read_epochs(
    '../outputs/filtered_data/S001_S010_64ch_combined-epo.fif',
    preload=True
)
print("Epochs loaded!")
print(epochs)

Reading /Users/kandulasatwika/Desktop/400_BCI/Notebooks/../outputs/filtered_data/S001_S010_64ch_combined-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =       0.00 ...    4000.00 ms
        0 CTF compensation matrices available
Not setting metadata
900 matching events found
No baseline correction applied
0 projection items activated
Epochs loaded!
<EpochsFIF | 900 events (all good), 0 – 4 s (baseline off), ~281.8 MB, data loaded,
 'rest': 450
 'left_fist': 227
 'right_fist': 223>


In [49]:
print("Number of channels:", len(epochs.info['ch_names']))
print("Total epochs:", len(epochs))

Number of channels: 64
Total epochs: 900


In [50]:
# Binary only — left vs right
from mne.decoding import CSP
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

epochs_binary = epochs[['left_fist', 'right_fist']]
X_bin = epochs_binary.get_data()
y_bin = epochs_binary.events[:, 2]

print("Left:", np.sum(y_bin==2))
print("Right:", np.sum(y_bin==3))

# CSP pipeline
csp_bin = CSP(n_components=4, reg=None, log=True)
svm_bin = SVC(kernel='linear', C=1.0, class_weight='balanced')

pipeline_bin = Pipeline([
    ('csp', csp_bin),
    ('svm', svm_bin)
])

scores_bin = cross_val_score(pipeline_bin, X_bin, y_bin, cv=5)
print(f"Binary CSP + SVM Accuracy: {scores_bin.mean()*100:.2f}%")

Left: 227
Right: 223
Computing rank from data with rank=None
    Using tolerance 0.00067 (2.2e-16 eps * 64 dim * 4.7e+10  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 0.00067 (2.2e-16 eps * 64 dim * 4.7e+10  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 0.00073 (2.2e-16 eps * 64 dim * 5.1e+10  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2

In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bin, y_bin,
    test_size=0.2,
    random_state=42,
    stratify=y_bin
)

pipeline_bin.fit(X_train, y_train)
y_pred = pipeline_bin.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Final Binary Accuracy: {accuracy * 100:.2f}%")

Computing rank from data with rank=None
    Using tolerance 0.00067 (2.2e-16 eps * 64 dim * 4.7e+10  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Final Binary Accuracy: 60.00%


In [52]:
# Approach 1 — Confidence threshold for REST
# Need probability scores so change SVM to use probability=True
csp_bin = CSP(n_components=4, reg=None, log=True)
svm_bin = SVC(kernel='linear', C=1.0, class_weight='balanced', probability=True)

pipeline_bin = Pipeline([
    ('csp', csp_bin),
    ('svm', svm_bin)
])

# Train
pipeline_bin.fit(X_train, y_train)

# Predict with confidence
y_prob = pipeline_bin.predict_proba(X_test)

commands = []
for prob in y_prob:
    max_confidence = np.max(prob)
    prediction = np.argmax(prob)
    
    if max_confidence < 0.6:
        command = "REST"
    elif prediction == 0:
        command = "LEFT"
    else:
        command = "RIGHT"
    
    commands.append(command)

# Show results
for i, (cmd, true) in enumerate(zip(commands, y_test)):
    true_label = "LEFT" if true == 2 else "RIGHT"
    print(f"Sample {i+1}: Predicted={cmd}, Actual={true_label}")

Computing rank from data with rank=None
    Using tolerance 0.00067 (2.2e-16 eps * 64 dim * 4.7e+10  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Sample 1: Predicted=LEFT, Actual=RIGHT
Sample 2: Predicted=LEFT, Actual=LEFT
Sample 3: Predicted=RIGHT, Actual=RIGHT
Sample 4: Predicted=RIGHT, Actual=RIGHT
Sample 5: Predicted=REST, Actual=LEFT
Sample 6: Predicted=RIGHT, Actual=RIGHT
Sample 7: Predicted=REST, Actual=RIGHT
Sample 8: Predicted=LEFT, Actual=LEFT
Sample 9: Predicted=REST, Actual=RIGHT
Sample 10: Predicted=LEFT, Actual=LEFT
Sample 11: Predicted=REST, Actual=RIGHT
Sample 12: Predicted=REST, Actual=LEFT
Sample 13: Predicted=REST, Actual=LEFT
Sample 14: Predicted=RIGHT, Actual=RIGHT
Sample 15: Predicted=REST, Actual=RIGHT
Sample 16: Predicted=LEFT, Actual=RIGHT
Sample 17: Predi